[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH07/07_01_end/notebooks/07_01_export_for_ollama.ipynb)

# 07_01 Merge your model and run it locally with Ollama

Merge your LoRA adapter into the base model, export it as a single GGUF file, and package it for Ollama.

Before you start, choose **Runtime > Change runtime type > T4 GPU**.

If you're using the instructor's model, you can skip this notebook. On your computer, go to the `CH07` folder of the course repo and run `ollama pull hf.co/KWillz/ridgeline-support-gguf`, then `ollama create ridgeline -f ollama/Modelfile.hf`. The Modelfile adds the chat template the model trained on.

In [ ]:
%%capture
# Install Unsloth. This takes a minute or two.
!pip install unsloth

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH07")

import ridgeline

## Choose your adapter

In [ ]:
# Set this to False if you didn't save your own adapter in Chapter 5.
USE_MY_ADAPTER = True

MY_ADAPTER = "/content/drive/MyDrive/ridgeline-lora"
INSTRUCTOR_ADAPTER = "KWillz/ridgeline-support-lora"

if USE_MY_ADAPTER:
    from google.colab import drive
    drive.mount("/content/drive")
    adapter = MY_ADAPTER
else:
    adapter = INSTRUCTOR_ADAPTER
print(f"Using adapter: {adapter}")

## Load the fine-tuned model

In [ ]:
from unsloth import FastLanguageModel

# Loading an adapter also loads the base model it was trained on.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter,
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.for_inference(model)

## Make sure it still works

In [ ]:
tickets = {t["id"]: t for t in ridgeline.load_tickets()}
print(ridgeline.generate(model, tokenizer, ridgeline.build_messages(tickets["RT-001"], include_voice_rules=False)))

## Merge and export

Merging folds the LoRA weights into the base model, so you end up with one standalone model. GGUF is the single-file format Ollama runs, and q4_k_m compresses it to 4 bits so it fits comfortably on a laptop.

In [ ]:
# Merge the LoRA weights into the base model and save one 4-bit GGUF file.
# This takes 10 to 15 minutes.
model.save_pretrained_gguf("ridgeline-gguf", tokenizer, quantization_method="q4_k_m")

In [ ]:
from pathlib import Path

# Find the 4-bit file. Unsloth may also leave a larger 16-bit file behind.
candidates = [p for d in Path("/content").glob("ridgeline-gguf*") for p in d.rglob("*.gguf")]
q4 = [p for p in candidates if "q4_k_m" in p.name.lower()]
gguf_file = min(q4 or candidates, key=lambda p: p.stat().st_size)
print(f"{gguf_file}  ({gguf_file.stat().st_size / 1e9:.1f} GB)")

## Write the Modelfile

The Modelfile tells Ollama which file to load and how to wrap each conversation. This template matches the chat format the model trained on.

In [ ]:
MODELFILE = '''FROM ./ridgeline.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0
PARAMETER num_ctx 2048
'''
print(MODELFILE)

## Save to Google Drive

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")
out = Path("/content/drive/MyDrive/ridgeline-ollama")
out.mkdir(parents=True, exist_ok=True)

shutil.copy(gguf_file, out / "ridgeline.gguf")
(out / "Modelfile").write_text(MODELFILE)
print(f"Saved ridgeline.gguf and Modelfile to {out}")

## Optional: publish to Hugging Face

In [ ]:
# Optional: publish your GGUF to your own Hugging Face account so you can pull it straight into Ollama.
# Needs an HF_TOKEN with write access in Colab Secrets.
# It uploads the exact file you saved to Drive, so everyone runs the model you tested.
PUBLISH = False
HF_USERNAME = "your-hf-username"  # change this to your Hugging Face username

if PUBLISH:
    from google.colab import userdata
    from huggingface_hub import HfApi

    api = HfApi(token=userdata.get("HF_TOKEN"))
    repo = f"{HF_USERNAME}/ridgeline-support-gguf"
    api.create_repo(repo, exist_ok=True)
    api.upload_file(
        path_or_fileobj="/content/drive/MyDrive/ridgeline-ollama/ridgeline.gguf",
        path_in_repo="ridgeline.gguf",
        repo_id=repo,
    )
    print(f"Published to https://huggingface.co/{repo}")

## Run it on your computer

1. Install Ollama from [ollama.com/download](https://ollama.com/download).
2. Download the `ridgeline-ollama` folder from your Google Drive.
3. In a terminal, go into that folder and run:

```
ollama create ridgeline -f Modelfile
ollama run ridgeline
```

4. In your local copy of the course repo, go to the `CH07` folder and run `python 07_01_end/scripts/ask_ridgeline.py RT-008` to send it a real ticket.